## Portfolio archive and evaluation note

This notebook is retained as part of the original MRI denoising experiment suite. Saved cells and outputs are included so the development work can be reviewed without rerunning long GPU training jobs.

> **Evaluation limitation:** In the supplied workflow, several experiments use the same image directory for both training and validation. The stored metrics are therefore historical development results, not independently reproduced estimates from a held-out validation set. A reliable rerun should use patient-level train, validation, and test splits.

The code was developed in Google Colab and contains hard-coded paths under `/content/drive/MyDrive/Final RI/`. Update those paths before running it with your own dataset.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import os
from IPython.display import Image, display

# =====================================================
# USER SETTINGS
# =====================================================

SAMPLE_INDEX = 5   # ← same sample for all models

EXPERIMENT_NAME = "Rician Noise (σ = 0.03)"
DATASET_NAME = "Glioma MRI Dataset"

# ---------- Proposed ----------
PROP_METRICS = "/content/drive/MyDrive/Final RI/results/rician_sigma_0.03/metrics_rician_sigma_0.03.csv"
PROP_DENOISED = "/content/drive/MyDrive/Final RI/results/rician_sigma_0.03/denoised_images"
PROP_NOISY = "/content/drive/MyDrive/Final RI/single_noise_outputs/single_noise_outputs_rician_sigma_0.03"

# ---------- No VAE ----------
NOVAE_METRICS = "/content/drive/MyDrive/Final RI/results/without_vae_rician_0.03/metrics_without_vae_rician_0.03.csv"
NOVAE_DENOISED = "/content/drive/MyDrive/Final RI/results/without_vae_rician_0.03/denoised_images"
NOVAE_NOISY = "/content/drive/MyDrive/Final RI/single_noise_outputs/without_vae_rician_0.03"
NOVAE_LOG = os.path.join(NOVAE_NOISY, "noise_log.csv")

# ---------- No DWT ----------
NODWT_METRICS = "/content/drive/MyDrive/Final RI/results/without_dwt_rician_0.03/metrics_without_dwt_rician_0.03.csv"
NODWT_DENOISED = "/content/drive/MyDrive/Final RI/results/without_dwt_rician_0.03/denoised_images"
NODWT_NOISY = "/content/drive/MyDrive/Final RI/single_noise_outputs/without_dwt_rician_0.03"
NODWT_LOG = os.path.join(NODWT_NOISY, "noise_log.csv")

# Ground truth
CLEAN_DIR = "/content/drive/MyDrive/Final RI/glioma"

SAVE_DIR = "/content/drive/MyDrive/Final RI/results/rician_ablation_comparison"
os.makedirs(SAVE_DIR, exist_ok=True)

# =====================================================
# HELPERS
# =====================================================

def load_gray(path):
    if not os.path.exists(path):
        print("Missing:", path)
        return None
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print("Could not load:", path)
    return img

def get_sorted_filename(directory, idx):
    files = sorted([
        f for f in os.listdir(directory)
        if f.lower().endswith((".png",".jpg",".jpeg"))
    ])
    return files[idx]

def get_filename_from_log(log_path, idx):
    log_df = pd.read_csv(log_path)
    return log_df.loc[idx, "filename"]

def get_metrics(path, idx):
    df = pd.read_csv(path)
    return df.loc[idx, "SSIM"], df.loc[idx, "PSNR"]

# =====================================================
# GET SAME SAMPLE NAME
# =====================================================

filename = get_sorted_filename(PROP_NOISY, SAMPLE_INDEX)

# =====================================================
# LOAD IMAGES
# =====================================================

clean_img = load_gray(os.path.join(CLEAN_DIR, filename))

# Proposed
prop_noisy = load_gray(os.path.join(PROP_NOISY, filename))
prop_denoised = load_gray(
    os.path.join(PROP_DENOISED, f"denoised_{SAMPLE_INDEX:03d}.png")
)

# No VAE
novae_filename = get_filename_from_log(NOVAE_LOG, SAMPLE_INDEX)
novae_noisy = load_gray(os.path.join(NOVAE_NOISY, novae_filename))
novae_denoised = load_gray(
    os.path.join(NOVAE_DENOISED, f"denoised_{SAMPLE_INDEX:03d}.png")
)

# No DWT
nodwt_filename = get_filename_from_log(NODWT_LOG, SAMPLE_INDEX)
nodwt_noisy = load_gray(os.path.join(NODWT_NOISY, nodwt_filename))
nodwt_denoised = load_gray(
    os.path.join(NODWT_DENOISED, f"denoised_{SAMPLE_INDEX:03d}.png")
)

# Safety check
if any(x is None for x in [
    clean_img,
    prop_noisy, prop_denoised,
    novae_noisy, novae_denoised,
    nodwt_noisy, nodwt_denoised
]):
    raise ValueError("Some images missing. Check paths.")

# =====================================================
# LOAD METRICS
# =====================================================

prop_ssim, prop_psnr = get_metrics(PROP_METRICS, SAMPLE_INDEX)
novae_ssim, novae_psnr = get_metrics(NOVAE_METRICS, SAMPLE_INDEX)
nodwt_ssim, nodwt_psnr = get_metrics(NODWT_METRICS, SAMPLE_INDEX)

# =====================================================
# SETUP FIGURE FOR LAYOUT
# =====================================================

fig, axes = plt.subplots(3, 3, figsize=(13, 12))

models = [
    ("Proposed Model (Full Architecture)", prop_noisy, prop_denoised, prop_ssim, prop_psnr),
    ("Ablation w/o VAE", novae_noisy, novae_denoised, novae_ssim, novae_psnr),
    ("Ablation w/o DWT", nodwt_noisy, nodwt_denoised, nodwt_ssim, nodwt_psnr)
]

col_titles = ["Ground Truth", "Noisy Input", "Denoised Output"]

for row, (model_name, noisy, denoised, ssim, psnr) in enumerate(models):
    images = [clean_img, noisy, denoised]

    # ---- Model title ----
    axes[row, 1].text(
        0.5, 1.3, model_name,
        transform=axes[row, 1].transAxes,
        ha="center",
        fontsize=15,
        weight="bold"
    )

    # ---- Column titles ----
    for col, title in enumerate(col_titles):
        axes[row, col].text(
            0.5, 1.1, title,
            transform=axes[row, col].transAxes,
            ha="center",
            fontsize=12,
            weight="bold"
        )

    # ---- Show images ----
    for col in range(3):
        axes[row, col].imshow(images[col], cmap="gray", vmin=0, vmax=255)
        axes[row, col].axis("off")

    # ---- Metrics below images ----
    axes[row, 1].text(
        0.5, -0.15,
        f"SSIM = {ssim:.4f}   |   PSNR = {psnr:.2f} dB",
        transform=axes[row, 1].transAxes,
        ha="center",
        fontsize=12
    )

plt.tight_layout(rect=[0, 0, 1, 0.95])

# =====================================================
# SAVE
# =====================================================

save_path = os.path.join(
    SAVE_DIR,
    f"rician_ablation_sigma003_sample{SAMPLE_INDEX}_layout_style.png"
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.close()

print(" Saved:", save_path)
display(Image(filename=save_path))

Output hidden; open in https://colab.research.google.com to view.